# Qwen2.5-7B Reward Model Reproduction — Colab Notebook

Reproduces the 7B experiments for the IF_RLHF paper on Google Colab A100.

## Before you start
1. Runtime → Change runtime type → **A100 GPU** (Pro+) or **L4** (Pro, slower)
2. Upload `IntroLLM_for_colab.tar.gz` to your Google Drive under `MyDrive/Colab/IntroLLM/`
3. Run cells in order. All intermediate results save to Drive so you can resume if disconnected.

## Step 0: Check GPU

In [ ]:
!nvidia-smi

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Setup paths
import os
DRIVE_DIR = '/content/drive/MyDrive/Colab/IntroLLM'
WORK_DIR = '/content/IntroLLM'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive: {DRIVE_DIR}')
print(f'Work: {WORK_DIR}')
!ls -la '{DRIVE_DIR}' 2>/dev/null || echo 'Drive folder is empty'

## Step 2: Extract project
Unpacks `IntroLLM_for_colab.tar.gz` from Drive into `/content/IntroLLM/`.
Also links persistent folders to Drive so checkpoints/gradients survive disconnections.

In [ ]:
# Extract project
if not os.path.exists(WORK_DIR):
    !tar -xzf '{DRIVE_DIR}/IntroLLM_for_colab.tar.gz' -C /content/
    !mv /content/IntroLLM* /content/IntroLLM 2>/dev/null
    print('Project extracted')
else:
    print('Project already extracted')

# Move logs to Drive so they persist across disconnections
DRIVE_LOGS = f'{DRIVE_DIR}/logs_7B'
os.makedirs(DRIVE_LOGS, exist_ok=True)

if not os.path.islink(f'{WORK_DIR}/logs_7B'):
    !rm -rf '{WORK_DIR}/logs_7B'
    !ln -s '{DRIVE_LOGS}' '{WORK_DIR}/logs_7B'
    print('Linked logs_7B → Drive')

%cd {WORK_DIR}
!ls

## Step 3: Install dependencies
Colab already has PyTorch + transformers. Install what's missing.

In [ ]:
!pip install -q peft>=0.12 trl>=0.10 bitsandbytes accelerate>=0.33
!pip install -q scikit-learn matplotlib numpy scipy pyyaml

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Step 4: HuggingFace login (optional, for gated models)
Only needed for Llama. Qwen models are public.

In [ ]:
# from huggingface_hub import login
# login()  # uncomment and paste your token

## Step 5: Configure paths & environment

In [ ]:
import os
os.environ['PYTHONPATH'] = f'{WORK_DIR}:{os.environ.get("PYTHONPATH","")}'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Modify configs to output to Drive-linked logs_7B
import yaml
for bias in ['length', 'sycophancy']:
    config_path = f'{WORK_DIR}/configs/reward_model_{bias}.yaml'
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    cfg['output_dir'] = f'logs_7B/Qwen2.5-7B_{bias}'
    # Reduce batch complexity for A100 40GB if needed
    cfg['gradient_checkpointing'] = False  # A100 has enough VRAM
    with open(config_path, 'w') as f:
        yaml.safe_dump(cfg, f)
    print(f'Updated {config_path}')

## Step 6: Train Length Bias RM
~1.5h on A100. Checkpoints save to Drive; if disconnected, just re-run.

In [ ]:
%cd {WORK_DIR}
!python src/reward_modeling/train.py --config configs/reward_model_length.yaml

## Step 7: Train Sycophancy Bias RM
~1.5h on A100.

In [ ]:
%cd {WORK_DIR}
!python src/reward_modeling/train.py --config configs/reward_model_sycophancy.yaml

## Step 8: Backup adapters (safety)
Copy the final adapter files from checkpoint if not auto-saved.

In [ ]:
import glob, shutil
for bias in ['length', 'sycophancy']:
    model_dir = f'{WORK_DIR}/logs_7B/Qwen2.5-7B_{bias}'
    # Check if adapter_model.safetensors exists
    if not os.path.exists(f'{model_dir}/adapter_model.safetensors'):
        ckpts = sorted(glob.glob(f'{model_dir}/checkpoint-*'))
        if ckpts:
            last_ckpt = ckpts[-1]
            print(f'Copying adapter from {last_ckpt}')
            for f in ['adapter_config.json', 'adapter_model.safetensors']:
                shutil.copy(f'{last_ckpt}/{f}', model_dir)
    print(f'{model_dir} adapter ready:', os.path.exists(f'{model_dir}/adapter_model.safetensors'))

## Step 9: Cache Gradients (Length Bias)
~2.5h on A100. Uses 4-bit quantization (~8GB VRAM).

In [ ]:
%cd {WORK_DIR}
!python -m src.influence.cache_gradients \
    --model_path logs_7B/Qwen2.5-7B_length \
    --tokenizer_path Qwen/Qwen2.5-7B-Instruct \
    --data_path dataset/length_dataset_tokenized/train \
    --save_name rapid_grad_train.pt --K 65536

!python -m src.influence.cache_gradients \
    --model_path logs_7B/Qwen2.5-7B_length \
    --tokenizer_path Qwen/Qwen2.5-7B-Instruct \
    --data_path dataset/length_dataset_tokenized/test \
    --save_name rapid_grad_val.pt --K 65536

## Step 10: Cache Gradients (Sycophancy Bias)

In [ ]:
%cd {WORK_DIR}
!python -m src.influence.cache_gradients \
    --model_path logs_7B/Qwen2.5-7B_sycophancy \
    --tokenizer_path Qwen/Qwen2.5-7B-Instruct \
    --data_path dataset/sycophancy_dataset_tokenized/train \
    --save_name rapid_grad_train.pt --K 65536

!python -m src.influence.cache_gradients \
    --model_path logs_7B/Qwen2.5-7B_sycophancy \
    --tokenizer_path Qwen/Qwen2.5-7B-Instruct \
    --data_path dataset/sycophancy_dataset_tokenized/test \
    --save_name rapid_grad_val.pt --K 65536

## Step 11: Compute Influence Scores

In [ ]:
%cd {WORK_DIR}
for bias in ['length', 'sycophancy']:
    !python -m src.influence.compute_influence \
        --model_path logs_7B/Qwen2.5-7B_{bias} \
        --dataset_dir dataset/{bias}_dataset \
        --bias_type {bias} --K 65536

## Step 12: View Results

In [ ]:
import json
print('=== 7B Results ===\n')
for bias in ['length', 'sycophancy']:
    path = f'{WORK_DIR}/logs_7B/Qwen2.5-7B_{bias}/influence_results.json'
    if os.path.exists(path):
        print(f'--- {bias} ---')
        print(json.dumps(json.load(open(path)), indent=2))
        print()

## Step 13: Download results to your local machine
The results are already on Drive (thanks to the symlink in Step 2).
You can download them via:
- Google Drive web interface → `MyDrive/Colab/IntroLLM/logs_7B/`
- Or use `rclone` / Drive desktop sync

Key files to copy back:
- `logs_7B/Qwen2.5-7B_length/influence_results.json`
- `logs_7B/Qwen2.5-7B_length/influence_*.npy`
- `logs_7B/Qwen2.5-7B_sycophancy/influence_results.json`
- `logs_7B/Qwen2.5-7B_sycophancy/influence_*.npy`

## Optional: Anti-Idle Heartbeat
Run this in a separate cell to prevent Colab from disconnecting during long runs.

In [ ]:
# DON'T run this along with training — only to keep connection alive after training finishes
# import time
# while True:
#     time.sleep(60)
#     print('.', end='', flush=True)